# **Data Cleaning and Prepocessing**

In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

## **Handling Missing Data**

Several machine learning models assume that there is no missing data. Some strategies to cope with this include:

- **Dropping rows with missing values:** Removing rows with missing data. When there are many missing values this can lead to a significant drop in available rows.
- **Removing sparse columns:** Columns with excessive missing data are often removed (e.g. 50% missing).
- **Imputation:** Filling in missing values with appropriate substitutes. For numeric values this is often the mean, median or 0 while for categorical ones it is often the mode (most common value) or a placeholder category e.g. "NA".

In [4]:
# Sample data with missing values
data = {'A': [1, 2, np.nan, 4],
        'B': [5, np.nan, np.nan, 8],
        'C': ['a', 'b', 'c', np.nan],
        'D': [1, np.nan, np.nan, np.nan]}

df = pd.DataFrame(data)
df

,A,B,C,D
0,1.0,5.0,a,1.0
1,2.0,NaN,b,NaN
2,NaN,NaN,c,NaN
3,4.0,8.0,NaN,NaN


In [5]:
# Dropping rows with missing values
df_row_dropped = df.dropna()
df_row_dropped

,A,B,C,D
0,1.0,5.0,a,1.0


In [6]:
# Impute missing values with 0
df_filled = df.fillna(0)
df_filled

,A,B,C,D
0,1.0,5.0,a,1.0
1,2.0,0.0,b,0.0
2,0.0,0.0,c,0.0
3,4.0,8.0,0,0.0


In [7]:
# Identify columns with more than 50% missing values (sparse columns)
threshold = 0.5 * len(df)
sparse_columns = df.columns[df.isnull().sum() > threshold]

# Drop sparse columns
df_col_dropped = df.drop(columns=sparse_columns)
df_col_dropped

,A,B,C
0,1.0,5.0,a
1,2.0,NaN,b
2,NaN,NaN,c
3,4.0,8.0,NaN


In [8]:
#Filling missing values Only For Specified Columns
df.fillna({'A':200,'C': 100})

,A,B,C,D
0,1.0,5.0,a,1.0
1,2.0,NaN,b,NaN
2,200.0,NaN,c,NaN
3,4.0,8.0,100,NaN


In [9]:
#filling with mean
x = df["A"].mean()
df.fillna({"A": x})

,A,B,C,D
0,1.000000,5.0,a,1.0
1,2.000000,NaN,b,NaN
2,2.333333,NaN,c,NaN
3,4.000000,8.0,NaN,NaN


## **Handling Outliers**

Common strategies to address outliers:

- **Removing outliers:** Using statistical methods like Z-scores, interquartile range (IQR), Mahalanobis Distance (multi-dimensional) to identify outliers for removal.
- **Transforming dataset:** Rescale the dataset to e.g. logarithmic scale reduce the overall impact of outliers.

In [10]:
# Sample data
data = {'TotalCharges': [100, 200, 3000, 400, 5000, 600, 7000, 800, 100000]}
df = pd.DataFrame(data)
df

,TotalCharges
0,100
1,200
2,3000
3,400
4,5000
5,600
6,7000
7,800
8,100000


In [11]:
#Handling Outliers with Z-Scores

# Calculate Z-scores
zscores = np.abs(stats.zscore(df['TotalCharges']))

# Filter out outliers
df_no_outliers = df[zscores < 2] 
df_no_outliers


,TotalCharges
0,100
1,200
2,3000
3,400
4,5000
5,600
6,7000
7,800


Z-Scores are best used when the data follows a normal distribution and when not too many or too extreme outliers are expected. They are good at identifying subtle outliers.

In [12]:
#Handling Outliers with Interquartile Range (IQR)

# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = df['TotalCharges'].quantile(0.25)
Q3 = df['TotalCharges'].quantile(0.75)

# Calculate IQR
IQR = Q3 - Q1

# Define the lower and upper bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers
df_no_outliers = df[(df['TotalCharges'] >= lower_bound) & (df['TotalCharges'] <= upper_bound)]

df_no_outliers

,TotalCharges
0,100
1,200
2,3000
3,400
4,5000
5,600
6,7000
7,800


IQR is robust so it can handle non-normal data and many extreme outliers well. It is less suitable to identify less extreme values that are still anomalies.

In [13]:
#Transforming Outliers with Log Transformation

# Sample data
data = {'TotalCharges': [-10, 100, 200, 3000, 400, 5000, 600, 7000, 800, 100000]}
df = pd.DataFrame(data)

# Log transform 'TotalCharges' to reduce the effect of outliers
df['Log_TotalCharges'] = np.log(df['TotalCharges'] + 11) 
df['Original_TotalCharges'] = round(np.exp(df['Log_TotalCharges'])) - 11 # Convert back to original values
df

,TotalCharges,Log_TotalCharges,Original_TotalCharges
0,-10,0.000000,-10.0
1,100,4.709530,100.0
2,200,5.351858,200.0
3,3000,8.010028,3000.0
4,400,6.018593,400.0
5,5000,8.519391,5000.0
6,600,6.415097,600.0
7,7000,8.855236,7000.0
8,800,6.698268,800.0
9,100000,11.513035,100000.0


This technique does not remove outliers but transforms the entire variable to make the distribution more normal-like. It allows keeping outliers while reducing their negative impact. Therefore it is suitable for data with heavy skew. Since log is undefined for negative values and infinite for 0, a constant might have to be added to make the variable positive.

## **Handling Duplicates**

In [14]:
# Sample data with duplicates
data = {'A': [1, 2, 2, 4],
        'B': [5, 6, 6, 8],
        'C': ['a', 'b', 'b', 'd']}
df = pd.DataFrame(data)
df

,A,B,C
0,1,5,a
1,2,6,b
2,2,6,b
3,4,8,d


In [15]:
# Identify duplicate rows
duplicates = df.duplicated() 
duplicates

0    False
1    False
2     True
3    False
dtype: bool

In [17]:
# Remove duplicate rows, keeping the first occurrence
df_no_duplicates = df.drop_duplicates()
df_no_duplicates

,A,B,C
0,1,5,a
1,2,6,b
3,4,8,d


## **Handling Data Inconsistency**
Inconsistent data occurs when the same information is represented differently across records. This can include inconsistent capitalisation, abbreviations, or formatting (e.g., 'NY' vs. 'New York'). To ensure consistency:

- **Standardise formats:** Ensure consistent representations.
- **Correct inconsistencies:** Apply uniform formats using methods like string normalisation.

In [18]:
# Sample data with inconsistent representations
data = {'City': ['New York', 'new york', 'NY', 'nyc']}
df = pd.DataFrame(data)
df

,City
0,New York
1,new york
2,NY
3,nyc


In [20]:
# Standardize the 'City' column by converting all values to lowercase
df['City_Standardized'] = df['City'].str.lower()
df

,City,City_Standardized
0,New York,new york
1,new york,new york
2,NY,ny
3,nyc,nyc


In [21]:
# Replace abbreviations with full names
df['City_Standardized'] = df['City_Standardized'].replace({'ny': 'new york', 'nyc': 'new york'})
df

,City,City_Standardized
0,New York,new york
1,new york,new york
2,NY,new york
3,nyc,new york


## **Correcting Data Types**
Typical corrections involve casting string dates and times to timestamps, string numbers to numerics or binary integers to booleans.

In [23]:
# Sample data with incorrect data types
data = {'Age': ['26', '32', '22'], 'DateOfBirth': ['1999-05-21', '1992-02-14', '2002-07-30']}
df = pd.DataFrame(data)
df

,Age,DateOfBirth
0,26,1999-05-21
1,32,1992-02-14
2,22,2002-07-30


In [24]:
#viewing data types of each column
print(df.dtypes)

Age            str
DateOfBirth    str
dtype: object


In [26]:
# Convert 'Age' from string to numeric
df['Age'] = pd.to_numeric(df['Age'])
print(df.dtypes)

Age            int64
DateOfBirth      str
dtype: object


In [28]:
# Convert 'DateOfBirth' to datetime
df['DateOfBirth'] = pd.to_datetime(df['DateOfBirth'])
print(df.dtypes)

Age                     int64
DateOfBirth    datetime64[us]
dtype: object


## **Encoding Categorical Features**
Machine learning algorithms typically require numerical inputs, so categorical features must be encoded. Two common encoding methods include:

- **One-hot encoding:** Converts each category into a new binary column. It is typically used when there is no inherent ordering between the categories and when there is a small number of categories. With many categories the dataset can quickly suffer from high-dimensionality.
- **Label encoding:** Assigns a unique integer to each category. This is more suitable when there are many categories. However, it imposes a numerical relationship onto the categories which can affect interpretability and predictive performance.

In [31]:
from sklearn.preprocessing import LabelEncoder

In [33]:
# Sample data
data = {'Category': ['A', 'B', 'A', 'C']}
df = pd.DataFrame(data)
df

,Category
0,A
1,B
2,A
3,C


In [36]:
# One-hot encode categorical features
df_one_hot = pd.get_dummies(df, columns=['Category'])
df_one_hot

,Category_A,Category_B,Category_C
0,True,False,False
1,False,True,False
2,True,False,False
3,False,False,True


In [37]:
# Label encode categorical features
label_encoder = LabelEncoder()
df['Category_LabelEncoded'] = label_encoder.fit_transform(df['Category'])
df

,Category,Category_LabelEncoded
0,A,0
1,B,1
2,A,0
3,C,2


## **Standardisation and Normalisation**
Standardisation and normalisation are techniques used to adjust the scale of features, ensuring that all features contribute equally to the model.

- **Standardisation(Z-score scaling)** scales data so that it has a mean of 0 and a standard deviation of 1. It’s often used in algorithms that assume normally distributed data (e.g., linear regression, SVMs).
- **Normalisation** rescales the data to a fixed range, typically between 0 and 1. It’s commonly used when the goal is to ensure that all features are on the same scale but not necessarily normally distributed.

In [38]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [39]:
# Sample data
data = {'Feature1': [100, 200, 300], 'Feature2': [4000, 5000, 6000]}
df = pd.DataFrame(data)
df

,Feature1,Feature2
0,100,4000
1,200,5000
2,300,6000


In [40]:
# Standardization (mean = 0, standard deviation = 1)
scaler = StandardScaler()
df_standardized = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df_standardized

,Feature1,Feature2
0,-1.224745,-1.224745
1,0.000000,0.000000
2,1.224745,1.224745


In [41]:
# Normalization (range 0 to 1)
normalizer = MinMaxScaler()
df_normalized = pd.DataFrame(normalizer.fit_transform(df), columns=df.columns)
df_normalized

,Feature1,Feature2
0,0.0,0.0
1,0.5,0.5
2,1.0,1.0
